In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

In [13]:
# Separate numerical and categorical features
num_features = train_df.select_dtypes(include=['int64', 'float64']).columns
cat_features = train_df.select_dtypes(include=['object']).columns

# Fill NaN in numerical features with median
num_imputer = SimpleImputer(strategy="median")
train_df[num_features] = num_imputer.fit_transform(train_df[num_features])

# Fill NaN in categorical features with "Missing"
cat_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
train_df[cat_features] = cat_imputer.fit_transform(train_df[cat_features])

In [14]:
from sklearn.preprocessing import OneHotEncoder

# Apply One-Hot Encoding
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Transform categorical features
encoded_cat = encoder.fit_transform(train_df[cat_features])

# Convert back to DataFrame with proper column names
encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(cat_features))

# Concatenate numerical and encoded categorical features
train_df = pd.concat([train_df[num_features], encoded_cat_df], axis=1)

In [16]:
scaler = StandardScaler()
train_df[num_features] = scaler.fit_transform(train_df[num_features])

In [17]:
import numpy as np

# y: target (log-space if already transformed)
# If NOT transformed earlier, uncomment the next line to apply log1p now:
# train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

y = train_df['SalePrice']              # target (expected to be log1p-transformed)
X = train_df.drop(columns=['SalePrice'])  # features only

In [18]:
from sklearn.model_selection import train_test_split

# Hold-out split: 80% train / 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
from sklearn.linear_model import LinearRegression

# Fit baseline linear regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# Predict on validation set
y_pred = linreg.predict(X_valid)

In [20]:
from sklearn.metrics import r2_score
import numpy as np

# --- Metrics in log-space (consistent with training target) ---
# R^2: proportion of variance explained (higher is better, <=1)
r2_log = r2_score(y_valid, y_pred)

print("R^2 (log-space):", r2_log)

R^2 (log-space): 0.9064342643302838


In [22]:
from sklearn.metrics import mean_squared_error
import numpy as np

# RMSE in log-space (lower is better)
rmse_log = np.sqrt(mean_squared_error(y_valid, y_pred))

print("RMSE (log-space):", rmse_log)

RMSE (log-space): 0.3309132611719924


In [25]:
# --- Optional: transform back to original price scale ---
# Only do this if y was log1p-transformed
y_valid_price = np.expm1(y_valid)   # inverse of log1p
y_pred_price  = np.expm1(y_pred)

rmse_price = np.sqrt(mean_squared_error(y_valid_price, y_pred_price))
print("RMSE (original price):", rmse_price)

RMSE (original price): 0.9229496410903454


In [27]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Naive predictor: always predict the mean of y_train (log-space)
y_naive = np.full_like(y_valid, fill_value=y_train.mean())
rmse_naive = np.sqrt(mean_squared_error(y_valid, y_naive))
print("Naive RMSE (log-space):", rmse_naive)

Naive RMSE (log-space): 1.084974720423473


In [28]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

# Cross-validated negative RMSE in log-space (higher is better, so we take negative and flip sign)
cv_scores = cross_val_score(
    LinearRegression(),
    X, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)
print("CV RMSE (log-space), mean +/- std:", -cv_scores.mean(), "+/-", cv_scores.std())

CV RMSE (log-space), mean +/- std: 0.3926750141452772 +/- 0.0748915259603191
